# Trackers en tiempo real — demostración

**Qué hace.** Procesa un video **frame a frame, como si llegara de la cámara**. YOLO hace de "detector externo" y uno de los cuatro trackers de `trackers_tiempo_real/` sigue los objetos. Guarda el video anotado, mide cuántos milisegundos tarda cada etapa y, con UCMC, muestra la cámara que estima en línea (pitch y altura).

**Para qué sirve.** Es la plantilla para llevarlos a otro entorno. Allí el detector ya existe: hay que sustituir `detectar()` (PARTE 1) por el propio, respetando el formato `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles del frame original. Todo lo demás se queda igual.

| Tracker | Archivo | Qué lo distingue |
|---|---|---|
| `"propio"` | `tracker_propio.py` | Estabilización con H + Kalman + IoU (el `propio_kf` de la comparativa) |
| `"ucmc"` | `tracker_ucmc.py` | Asocia en el suelo (metros); la cámara se calibra sola, en línea |
| `"bytetrack"` | `tracker_bytetrack.py` | Referencia: Kalman en la imagen, sin compensar la cámara |
| `"botsort"` | `tracker_botsort.py` | ByteTrack + su propia compensación de cámara (GMC) |

La explicación completa (contrato de entrada y salida, pipelines, de manual a automático en UCMC) está en `trackers_tiempo_real/README.md`.

In [ ]:
# ======================================================================
# IMPORTACIONES
# Qué hace : Carga las librerías y la carpeta de los trackers
# ¿Se toca? : Solo CARPETA si mueves la carpeta
# ======================================================================
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

CARPETA = Path("trackers_tiempo_real")                 # <-- dónde están los .py de los trackers
sys.path.insert(0, str(CARPETA.resolve()))

from comun import Config, como_array, dibujar
from tracker_propio import TrackerPropio, ConfigPropio
from tracker_ucmc import TrackerUCMC, ConfigUCMC, CamaraUCMC
from tracker_bytetrack import TrackerByteTrack
from tracker_botsort import TrackerBoTSORT

## PARTE 1 — Detector externo
**Entra:** un frame BGR a resolución original. **Sale:** `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles de **ese** frame, y `nombres` = `{id: "car", ...}`.

Aquí se simula con YOLO, con los mismos modelos y umbrales de la comparativa: OBB (DOTA) para tomas cenitales y COCO para oblicuas. **En el otro entorno solo se reemplaza esta celda.**

In [ ]:
# ======================================================================
# PARTE 1 · Detector externo (simulado con YOLO)
# Qué hace : Devuelve las detecciones de un frame en el formato que esperan los trackers
# ¿Se toca? : SÍ — en el otro entorno, esta celda se sustituye por el detector que ya existe
# ======================================================================
from ultralytics import YOLO

class DetectorYOLO:
    def __init__(self, pesos, clases, imgsz=1280, conf=0.15):
        self.modelo = YOLO(pesos)
        inv = {str(v).lower(): k for k, v in self.modelo.names.items()}
        self.clases = [inv[c] for c in clases]
        self.nombres = dict(self.modelo.names)
        self.imgsz, self.conf = imgsz, conf

    def __call__(self, frame):
        r = self.modelo.predict(frame, imgsz=self.imgsz, conf=self.conf, classes=self.clases, verbose=False)[0]
        b = r.obb if getattr(r, "obb", None) is not None else r.boxes     # OBB: se usa su caja horizontal
        if b is None or len(b) == 0:
            return np.zeros((0, 6), np.float32)
        return np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None], b.cls.cpu().numpy()[:, None]], 1).astype(np.float32)

DETECTOR_CENITAL = lambda: DetectorYOLO("yolo11n-obb.pt", ("small vehicle", "large vehicle"), conf=0.15)
DETECTOR_OBLICUO = lambda: DetectorYOLO("yolo11n.pt", ("car", "truck", "bus", "motorcycle"), conf=0.25)

## PARTE 2 — Crear el tracker
Cada tracker es **una clase**, con la misma forma de uso:

```python
tracker = TrackerXXX(fps, cfg, nombres_clases)
salida = tracker.actualizar(frame, detecciones)
```

- **UCMC** necesita además los datos de la cámara (`CamaraUCMC`):
  - `hfov_deg`: dato de la ficha técnica del dron (82° por defecto). **No se estima.**
  - `pitch_respaldo_deg`: estimación a ojo. Solo se usa hasta que la H dé la primera medida, o siempre si el dron no se desplaza.
  - `altura_m=None`: altura automática; con un número, queda fija (por ejemplo, la de la telemetría).
- **Para un benchmark MOT:** `Config(use_proposals=False)`, para no sumar cajas sin clase.

In [ ]:
# ======================================================================
# PARTE 2 · crear_tracker
# Qué hace : Crea el tracker elegido con su configuración
# ¿Se toca? : No (lo que se elige está en el Paso B)
# ======================================================================
def crear_tracker(tipo, fps, nombres, camara=None, work_width=960, propuestas=True):
    if tipo == "propio":
        return TrackerPropio(fps, ConfigPropio(work_width=work_width, use_proposals=propuestas), nombres)
    if tipo == "ucmc":
        return TrackerUCMC(fps, camara or CamaraUCMC(), ConfigUCMC(work_width=work_width, use_proposals=propuestas), nombres)
    if tipo == "bytetrack":
        return TrackerByteTrack(fps, Config(work_width=work_width, use_proposals=propuestas), nombres)
    if tipo == "botsort":
        return TrackerBoTSORT(fps, Config(work_width=work_width, use_proposals=propuestas), nombres)
    raise ValueError(f"tracker desconocido: {tipo}")

## PARTE 3 — Bucle en tiempo real
Lee un frame, detecta, actualiza el tracker, dibuja y escribe. Es lo mismo que haría el otro entorno con la cámara en vivo.

Por cada frame guarda:
- los ms del detector;
- los ms de cada etapa del tracker (`tracker.tiempos_ms`);
- con UCMC, la cámara estimada (`tracker.calibrador`).

Así se ve si el tracker llega a tiempo real y cuál etapa pesa más.

In [ ]:
# ======================================================================
# PARTE 3 · procesar
# Qué hace : Bucle frame a frame (detector -> tracker -> dibujo) con medición de tiempos
# ¿Se toca? : No
# ======================================================================
def procesar(video, tipo, detector, camara=None, out_dir="salidas/tiempo_real", max_frames=None, out_width=1920, propuestas=True):
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    tracker = crear_tracker(tipo, fps, detector.nombres, camara, propuestas=propuestas)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    salida_mp4 = Path(out_dir) / f"{Path(video).stem}_{tipo}.mp4"
    writer, filas, mot = None, [], []
    k = 0
    while max_frames is None or k < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        dets = detector(frame)                                           # detector externo
        t_det = 1000 * (time.perf_counter() - t0)
        tracks = tracker.actualizar(frame, dets)                         # <- el tracker
        fila = dict(frame=k, detector_ms=round(t_det, 1), tracker_ms=round(sum(tracker.tiempos_ms.values()), 2),
                    **{f"{e}_ms": v for e, v in tracker.tiempos_ms.items()}, tracks=len(tracks),
                    en_movimiento=sum(t.en_movimiento for t in tracks))
        if tipo == "ucmc":
            c = tracker.calibrador
            fila.update(pitch=round(c.pitch, 2), altura_m=round(c.altura, 1), fuente_pitch=c.fuente_pitch, fuente_altura=c.fuente_altura)
        filas.append(fila)
        mot += [[k + 1, *r] for r in como_array(tracks)]                 # formato MOT: frame, x1, y1, x2, y2, id, conf, clase
        vis = dibujar(frame.copy(), tracks, tipo)
        if out_width:
            vis = cv2.resize(vis, (out_width, int(round(vis.shape[0] * out_width / vis.shape[1]))), interpolation=cv2.INTER_AREA)
        if writer is None:
            writer = cv2.VideoWriter(str(salida_mp4), cv2.VideoWriter_fourcc(*"mp4v"), fps, (vis.shape[1], vis.shape[0]))
        writer.write(vis)
        k += 1
    cap.release()
    if writer is not None:
        writer.release()
    pd.DataFrame(mot, columns=["frame", "x1", "y1", "x2", "y2", "id", "conf", "clase"]).to_csv(
        Path(out_dir) / f"{Path(video).stem}_{tipo}_tracks.csv", index=False)
    df = pd.DataFrame(filas)
    print(f"{tipo}: {k} frames | tracker {df.tracker_ms.median():.1f} ms/frame (p95 {df.tracker_ms.quantile(0.95):.1f}) | "
          f"detector {df.detector_ms.median():.0f} ms/frame | IDs creados: {getattr(tracker, 'ids_emitidos', 'n/a')} | {salida_mp4}")
    return df, tracker

## PARTE 4 — Ejecución
`PARTE 1` → `PARTE 3` solo definen cosas; aquí se ejecuta.

### Paso A — Self-tests (ejecutar primero, sobre todo en un entorno nuevo)

In [ ]:
import pruebas
pruebas.prueba_comun()
import tracker_propio, tracker_ucmc
tracker_propio.selftest()
tracker_ucmc.selftest()
pruebas.prueba_pipeline_completo()
pruebas.prueba_ultralytics()

### Paso B — Procesar un video
Elige el video, el tracker y, si es UCMC, la cámara. Para probar rápido usa `max_frames=300`.

Salidas en `salidas/tiempo_real/`:
- `<video>_<tracker>.mp4`: video anotado.
- `<video>_<tracker>_tracks.csv`: tracks en formato MOT, para el benchmark.

In [ ]:
VIDEO_PATH = Path("/Users/Alex/Documents/videos_VIGIA/vigia_8.mp4")     # <-- CAMBIAR por tu video
TRACKER = "ucmc"                                                       # "propio" | "ucmc" | "bytetrack" | "botsort"
detector = DETECTOR_OBLICUO()                                          # toma oblicua; cenital: DETECTOR_CENITAL()
CAMARA = CamaraUCMC(hfov_deg=82, pitch_respaldo_deg=50, nota="pitch a ojo; solo hasta la 1ª medida")   # solo UCMC

df, tracker = procesar(VIDEO_PATH, TRACKER, detector, camara=CAMARA, max_frames=None)
display(df.describe().T[["50%", "max"]].rename(columns={"50%": "mediana"}).round(2))

In [ ]:
from IPython.display import Video, display
display(Video(f"salidas/tiempo_real/{VIDEO_PATH.stem}_{TRACKER}.mp4", embed=False, width=1100))

### Paso C — UCMC: la cámara que estimó en línea
- **Pitch:** hasta la primera medida es el de respaldo (gris); después sale de la H (azul).
- **Altura:** al principio es la de defecto (100 m); después la fijan los vehículos y la H.

Si existe la calibración del v13 (`comparativa_v13/<video>/calibracion_ucmc.json`, que se hacía **sobre el video completo, antes de procesar**), se dibuja encima para comparar.

In [ ]:
import json
import matplotlib.pyplot as plt
if TRACKER == "ucmc":
    fps = cv2.VideoCapture(str(VIDEO_PATH)).get(cv2.CAP_PROP_FPS)
    t = df.frame / fps
    fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    resp = df.fuente_pitch == "respaldo"
    ax[0].plot(t, df.pitch, color="tab:blue", label="en línea (este tracker)")
    ax[0].plot(t[resp], df.pitch[resp], ".", color="0.6", ms=2, label="respaldo")
    ax[1].plot(t, df.altura_m, color="tab:orange", label="en línea (este tracker)")
    v13 = Path("..") / "comparativa_v13" / VIDEO_PATH.stem / "calibracion_ucmc.json"
    v13 = v13 if v13.exists() else Path("comparativa_v13") / VIDEO_PATH.stem / "calibracion_ucmc.json"
    if v13.exists():
        c = json.load(open(v13))
        n = min(len(c["pitch_serie"]), len(df))
        ax[0].plot(np.arange(n) / fps, c["pitch_serie"][:n], "--", color="k", lw=1, label="v13 (video completo, offline)")
        if c.get("altura_serie"):
            ax[1].plot(np.arange(n) / fps, c["altura_serie"][:n], "--", color="k", lw=1, label="v13 (video completo, offline)")
    ax[0].set_ylabel("pitch (°)"); ax[1].set_ylabel("altura (m)"); ax[1].set_xlabel("s")
    for a in ax:
        a.grid(alpha=0.3); a.legend(fontsize=8)
    ax[0].set_title(f"{VIDEO_PATH.stem}: cámara estimada por UCMC (hfov {CAMARA.hfov_deg}° fijo)")
    plt.show()

### Paso D — ¿Llega a tiempo real?
ms por etapa (mediana y p95) del tracker, sin contar el detector. A 30 fps hay 33 ms por frame **para todo**, detector incluido.

In [ ]:
etapas = [c for c in df.columns if c.endswith("_ms")]
display(df[etapas].describe(percentiles=[0.5, 0.95]).T[["50%", "95%", "max"]].round(2))